<a href="https://colab.research.google.com/github/reinaldoyungh/Python/blob/main/IA_CATEGORIZAR_COMENTARIOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
from groq import Groq
client = Groq()

import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.7 MB/s eta 0:00:00


In [2]:

def chama_IA(perguntas):
    all_responses = []
    for pergunta_individual in perguntas:
        response_stream = client.chat.completions.create(
          model="openai/gpt-oss-20b",
          messages=[
            {
              "role": "user",
              "content": pergunta_individual
            }
          ],
          temperature=1,
          max_completion_tokens=1024,
          top_p=1,
          reasoning_effort="medium",
          stream=True,
          stop=None
        )

        collected_text = ""
        for chunk in response_stream:
            if chunk.choices and chunk.choices[0].delta.content:
                collected_text += chunk.choices[0].delta.content
        all_responses.append(collected_text)
    return all_responses

In [4]:
lista_reviews = []
with open('reviews.csv', 'r') as arquivo:
  for linha in arquivo:
    lista_reviews.append(linha.strip())

In [5]:
import io

# Convert the list of CSV strings into a pandas DataFrame
df_reviews = pd.read_csv(io.StringIO('\n'.join(lista_reviews)))

# Display the 'reviewerName' and 'reviewText' columns
display(df_reviews[['reviewerName', 'reviewText']])




,reviewerName,reviewText
0,1K3,it works as expected. I should have sprung for...
1,1m2,This think has worked out great.Had a diff. br...
2,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit..."
3,2Cents!,It's mini storage. It doesn't do anything els...
4,2K1Toaster,I have it in my phone and it never skips a bea...
5,98020,"It works, but file writes are a bit slower tha..."
6,"Abdulrahman J. Alrashed ""dr34m3r""","I bought 2 of those SanDisk 32 GB microSD , us..."
7,"Abel Feliciano ""Ace Master""",The memory card is an excellent condition and ...
8,Abraham Arturo Meza Marin,I bougth this micro SD card after some trubles...
9,"Abused Commuter ""abused_commuter""",Ordered this for a Galaxy S3. Lasted a few mo...


In [7]:
reviews_to_classify = df_reviews['reviewText'].tolist()

# Prepare questions for the LLM to classify sentiment
perguntas_sentimento = [
    f"Classifique o sentimento da seguinte avaliação do cliente como Positivo, Negativo ou Neutro: {review}"
    for review in reviews_to_classify
]

# Call the LLM function to get sentiments
sentimentos = chama_IA(perguntas_sentimento)

# Add the sentiments as a new column to the DataFrame
df_reviews['sentimento'] = df_reviews['sentimento'].str.replace('**', '', regex=False).str.strip()

# Display the DataFrame with the new sentiment column
display(df_reviews[['reviewerName', 'reviewText', 'sentimento']].head())

,reviewerName,reviewText,sentimento
0,1K3,it works as expected. I should have sprung for...,Neutro
1,1m2,This think has worked out great.Had a diff. br...,Positivo
2,2&amp;1/2Men,"Bought it with Retail Packaging, arrived legit...",Positivo
3,2Cents!,It's mini storage. It doesn't do anything els...,Positivo
4,2K1Toaster,I have it in my phone and it never skips a bea...,Positivo


In [ ]:
df_reviews.to_csv('reviews_avaliados.csv', index=False)
print('DataFrame salvo como reviews_avaliados.csv')

In [12]:
df_reviews_negativas = df_reviews[df_reviews["sentimento"] == "Negativo"]
df_reviews_negativas

,reviewerID,asin,reviewerName,reviewText,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,sentimento
6,AF24M1HKIZ7QC,B007WTAJTO,"Abdulrahman J. Alrashed ""dr34m3r""","I bought 2 of those SanDisk 32 GB microSD , us...",1375488000,08/03/2013,640,0,0,Negativo
8,A1FKE13D77L3Y3,B007WTAJTO,Abraham Arturo Meza Marin,I bougth this micro SD card after some trubles...,1361232000,19/02/2013,657,0,0,Negativo
9,A1X1FX3NSOFCT3,B007WTAJTO,"Abused Commuter ""abused_commuter""",Ordered this for a Galaxy S3. Lasted a few mo...,1374278400,20/07/2013,506,0,1,Negativo
10,A3RV1L35IPTV61,B007WTAJTO,abushaheed,more like 8mb/s in my Note 10.1 and that's usi...,1357430400,01/06/2013,555,0,0,Negativo
11,A2RJ7DAL63MN1F,B007WTAJTO,Ace of Sevens,"I used this for a few months in my phone, then...",1391126400,31/01/2014,311,0,0,Negativo
14,APAEBFIM1588A,B007WTAJTO,"Adam Wright ""Electrical Engineer""",This card advertises itself as UHS1 but it's a...,1378684800,09/09/2013,455,1,1,Negativo
15,A1JOXGZH55JMHW,B007WTAJTO,A. Hejazi,I bought this SD card for my android phone thi...,1386806400,12/12/2013,361,0,0,Negativo
18,A28CTDM7OP0RAR,B007WTAJTO,Alan,It works but the actual R/W speeds are 18/8 MB...,1395705600,25/03/2014,258,0,0,Negativo
19,A1NA2JGI3GNDN2,B007WTAJTO,Albert,Does it's job and fairly cheap for what it's w...,1399161600,05/04/2014,247,0,0,Negativo
21,A1UM44ILLZCEI1,B007WTAJTO,Aleksandar Milivojevic,I bought 64GB version of the card to use with ...,1364947200,04/03/2013,644,1,1,Negativo


In [14]:
resenhas_negativas = df_reviews_negativas["reviewText"]
resenhas_negativas

,reviewText
6,"I bought 2 of those SanDisk 32 GB microSD , us..."
8,I bougth this micro SD card after some trubles...
9,Ordered this for a Galaxy S3. Lasted a few mo...
10,more like 8mb/s in my Note 10.1 and that's usi...
11,"I used this for a few months in my phone, then..."
14,This card advertises itself as UHS1 but it's a...
15,I bought this SD card for my android phone thi...
18,It works but the actual R/W speeds are 18/8 MB...
19,Does it's job and fairly cheap for what it's w...
21,I bought 64GB version of the card to use with ...


In [16]:
resenhas_negativas_unidas = "#####".join(resenhas_negativas)

In [24]:
resposta_categorias = [
    f"""Você é um analista de dados, vou te passar muitas resenhas negativas de analises de um produto e eu quero que você encontre cinco categorias distintas para os tipos de reclamações, em portugues. Quero que você me retorne as cinco categorias. Cada categoria deve ser definida APENAS uma palavra. Além disso não escreva nada mais além das cinco categoria. Quero que elas sejam retornadas separadas por víngulas e em letras minusculas e sem acentos graficos.
    Exemplos: 'eletronicos, roupas, alimentos, viagens, brinquedos'

    Aqui estão as resenhas negativas separadas por #####: {resenhas_negativas_unidas}"""
]

resposta_IA = chama_IA(resposta_categorias)
print (resposta_IA)

['durabilidade, velocidade, compatibilidade, corrupcao, engano']


In [27]:
lista_categorias = resposta_IA[0].split(sep=', ')
lista_categorias

['durabilidade', 'velocidade', 'compatibilidade', 'corrupcao', 'engano']